# Session 1: a fair test, and accurate versus precise
**A. Real-world problem.** You want to know whether a lid keeps a cup of hot water warmer.
If you put the lid on a big cup and leave a small cup bare, you have changed two things
at once, so you cannot tell which one mattered. And if you try it only once, luck decides.

**Plan.** (1) a made-up cooling test where the true answer is known, (2) change one thing
at a time, (3) repeat the test, (4) tell *accurate* (close to the truth) from *precise*
(repeatable).

In [ ]:
import math
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

# --- the made-up world: how many degrees C a cup of water cools in 10 minutes -------------
BASE_DROP = 20.0     # small cup, no lid
LID_EFFECT = -1.0    # a lid changes the drop by this much (degrees C): the thing we want to find
BIG_EFFECT = -3.0    # a big cup also changes the drop (degrees C): a thing we must NOT mix in
NOISE_SD = 1.5       # run-to-run scatter of one measurement (degrees C)

## Step 1: the world, and a test of it
`expected_drop` is the truth with no scatter. `run_test` adds the scatter that every real
measurement has (a random number generator stands in for the unavoidable small wobbles).

In [ ]:
def expected_drop(lid, big):
    """True average cooling drop (degrees C). lid and big are 0 or 1."""
    return BASE_DROP + LID_EFFECT * lid + BIG_EFFECT * big


def run_test(lid, big, rng):
    """One measured drop: the truth plus random scatter."""
    return expected_drop(lid, big) + rng.normal(0.0, NOISE_SD)

## Step 2: change one thing at a time
An *unfair* test compares "lid on a big cup" with "no lid on a small cup": the lid and the
cup size both change. A *fair* test compares the same cup with and without the lid.
With no scatter at all you can already see the difference:

    unfair: (20 - 1 - 3) - (20) = -4        fair: (20 - 1) - (20) = -1

In [ ]:
def unfair_effect():
    """Apparent lid effect when the lid goes on the big cup and the bare cup is small."""
    return expected_drop(1, 1) - expected_drop(0, 0)


def fair_effect():
    """Lid effect when only the lid changes (same cup)."""
    return expected_drop(1, 0) - expected_drop(0, 0)

## Step 3: do it more than once
One run of each setting gives a wobbly answer. The *mean* of n runs wobbles less: its
scatter is NOISE_SD / sqrt(n). `p_wrong_sign(n)` is the chance that a fair test with n
runs per setting says the lid makes things *worse* (a positive difference) when it really
helps. The difference of two means has scatter sqrt(2/n) * NOISE_SD; the chance of a wrong
sign is the normal tail beyond |effect| / that scatter.

In [ ]:
def normal_tail(z):
    """P(Z > z) for a standard normal Z."""
    return 0.5 * math.erfc(z / math.sqrt(2.0))


def p_wrong_sign(n):
    scatter = NOISE_SD * math.sqrt(2.0 / n)
    return normal_tail(abs(LID_EFFECT) / scatter)


def simulate_wrong_sign(n, trials, seed):
    """Fraction of simulated fair tests (n runs per setting) whose measured effect is > 0."""
    rng = np.random.default_rng(seed)
    with_lid = expected_drop(1, 0) + rng.normal(0, NOISE_SD, (trials, n)).mean(axis=1)
    bare = expected_drop(0, 0) + rng.normal(0, NOISE_SD, (trials, n)).mean(axis=1)
    return float(np.mean(with_lid - bare > 0))

## Step 4: accurate versus precise
A thermometer reads a cup whose true temperature is 50.0 degrees C.
*Accurate* = the readings average close to the truth (small **bias**, the average error).
*Precise* = the readings agree with each other (small **scatter**, the standard deviation).
These are different: a thermometer can be precise but wrong, or right on average but jumpy.

In [ ]:
TRUE_TEMP = 50.0


def thermometer(bias, sd, n, seed, true=TRUE_TEMP):
    """n readings: truth + bias + random scatter."""
    rng = np.random.default_rng(seed)
    return true + bias + rng.normal(0.0, sd, n)


def describe(readings, true=TRUE_TEMP):
    """(mean error = bias estimate, scatter = standard deviation of the readings)."""
    r = np.asarray(readings, dtype=float)
    return float(r.mean() - true), float(r.std(ddof=1))


# four instruments: (bias, sd) in degrees C
INSTRUMENTS = {
    "accurate and precise": (0.0, 0.1),
    "accurate, not precise": (0.0, 1.0),
    "precise, not accurate": (2.0, 0.1),
    "neither": (2.0, 1.0),
}

## Step 5: see it
Left: the four thermometers as target boards (each dot is one reading; the cross is the
truth). Right: how often a one-run test gets the wrong answer, against the number of runs.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    print("unfair test says", unfair_effect(), "; fair test says", fair_effect(), "degrees C")
    for n in (1, 3, 9, 25):
        print(f"n = {n:2d} runs per setting: wrong-sign chance {p_wrong_sign(n):.3f}"
              f" (simulated {simulate_wrong_sign(n, 20000, 1):.3f})")

    fig, axs = plt.subplots(1, 4, figsize=(10, 2.8), sharey=True)
    for ax, (name, (b, s)) in zip(axs, INSTRUMENTS.items()):
        rng = np.random.default_rng(4)
        x = rng.normal(b, s, 25)
        y = rng.normal(b, s, 25)
        ax.scatter(x, y, s=12)
        ax.plot([0], [0], "r+", ms=14, mew=2)
        ax.set_xlim(-4, 4)
        ax.set_ylim(-4, 4)
        ax.set_aspect("equal")
        ax.set_title(name, fontsize=8)
        ax.set_xlabel("error left-right (degrees C)")
    axs[0].set_ylabel("error up-down (degrees C)")
    fig.suptitle("Target boards: red cross = truth, dots = readings (x, y are two independent error readings)", fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "accurate_precise.png", dpi=120)

    ns = np.arange(1, 31)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(ns, [p_wrong_sign(int(k)) for k in ns], "o-", ms=3)
    ax2.set_xlabel("runs per setting, n")
    ax2.set_ylabel("chance the fair test gets the wrong sign")
    ax2.set_title("More repeats make the fair test trustworthy")
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "wrong_sign_vs_n.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** In `lesson.py` change LID_EFFECT to -3.0 and BIG_EFFECT to -1.0. Predict what the unfair and fair tests report, then run `unfair_effect()` and `fair_effect()`.
2. **Stretch.** Use `simulate_wrong_sign(n, 20000, 1)` for n = 2, 4, 16 and compare with `p_wrong_sign(n)`. Find the smallest n for which the chance is below 0.05.
3. **Explain.** Explain in your own words why a thermometer with bias 2.0 and sd 0.1 is better than one with bias 0 and sd 1.0 for one reading, but worse for 25 readings.

## Assessment
- **A. Conceptual.** A friend says: 'My plant food works: the plants I fed in the sunny window grew taller than the unfed ones on the shelf.' What is wrong with this test, and how would you fix it?
- **B. Numerical.** In the cooling test (lid effect -1.0 degrees C, scatter 1.5 degrees C per run) you use n = 9 runs per setting. What is the scatter of the difference of the two means, and the chance of getting the wrong sign?
- **C. Graph and simulation interpretation.** In the app graph view, bias is 2.0, sd is 1.0 and you move n from 1 to 100. Describe the black curve and what limits it.
- **D. Design and optimisation challenge.** Design: you have a thermometer with bias 2.0 degrees C and sd 0.1 degrees C, and another with bias 0 and sd 1.0 degrees C. You can afford at most 25 readings in total. Which do you choose to measure a temperature to within 0.5 degrees C, and what do you do with the first one?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import math

import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("unfair test (lid and cup size both change) says -4.0 degrees C",
                   abs(L.unfair_effect() - (-4.0)) < 1e-12))
    checks.append(("fair test (only the lid changes) says -1.0 degrees C",
                   abs(L.fair_effect() - (-1.0)) < 1e-12))
    checks.append(("the unfair test is wrong by the cup effect, -3.0 degrees C",
                   abs((L.unfair_effect() - L.fair_effect()) - L.BIG_EFFECT) < 1e-12))
    checks.append(("worked example: one run per setting gives the wrong sign with chance 0.319",
                   abs(L.p_wrong_sign(1) - 0.319) < 0.0005))
    checks.append(("hand calculation: n = 1 -> z = 1 / (1.5 * sqrt(2)) = 0.4714, tail 0.319",
                   abs(L.normal_tail(1.0 / (1.5 * math.sqrt(2.0))) - 0.319) < 0.0005))
    checks.append(("simulation agrees with the formula for n = 1 (within 0.01)",
                   abs(L.simulate_wrong_sign(1, 20000, 1) - L.p_wrong_sign(1)) < 0.01))
    checks.append(("9 runs per setting: wrong-sign chance 0.079 (simulated within 0.01)",
                   abs(L.p_wrong_sign(9) - 0.079) < 0.0005 and abs(L.simulate_wrong_sign(9, 20000, 1) - 0.079) < 0.01))
    checks.append(("25 runs per setting: wrong-sign chance below 0.01",
                   L.p_wrong_sign(25) < 0.01))
    checks.append(("wrong-sign chance never rises as n grows (n = 1..30)",
                   all(L.p_wrong_sign(k + 1) <= L.p_wrong_sign(k) for k in range(1, 30))))
    checks.append(("normal_tail(0) = 0.5 (edge case)", abs(L.normal_tail(0.0) - 0.5) < 1e-12))
    # accurate vs precise: 2000 readings, mean error close to bias, sd close to sd
    ok = True
    for name, (b, s) in L.INSTRUMENTS.items():
        me, sd = L.describe(L.thermometer(b, s, 2000, 7))
        ok &= abs(me - b) < 4 * s / math.sqrt(2000) and abs(sd - s) < 0.1 * s
    checks.append(("each thermometer's mean error is about its bias and its scatter about its sd", ok))
    me, sd = L.describe(L.thermometer(2.0, 0.1, 2000, 7))
    checks.append(("precise-but-not-accurate: mean error 2.0 +- 0.01, scatter about 0.1",
                   abs(me - 2.0) < 0.01 and abs(sd - 0.1) < 0.01))
    r = L.thermometer(0.0, 1.0, 100000, 11)
    checks.append(("averaging many readings of an unbiased jumpy thermometer approaches the truth",
                   abs(r.mean() - 50.0) < 0.02))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
